# Bengali NLP Safety — V17 Kaggle Dual-GPU (Fixed Input)

**Protected champion: 0.586**

This Kaggle version runs the two model families **in parallel**:
- GPU 0 → BanglaBERT, 5 folds
- GPU 1 → XLM-R, 5 folds

### Setup
1. Kaggle Notebook → **Settings → Accelerator → GPU T4 x2**
2. Turn **Internet ON** if available/allowed, because Hugging Face pretrained models must be downloaded.
3. Add/upload the supplied `v17_data.zip` as notebook input.
4. Run all.
5. Download `v17_results.zip` from `/kaggle/working`.
6. Send the ZIP to ChatGPT before any Kaggle submission.


In [ ]:
# 1. Locate V17 data on Kaggle — works with ZIP OR Kaggle-extracted dataset
from pathlib import Path
import zipfile, subprocess, sys, os, shutil

subprocess.run(["nvidia-smi"], check=False)

INPUT_ROOT = Path("/kaggle/input")
WORK_DATA = Path("/kaggle/working/v17_data")
WORK_DATA.mkdir(parents=True, exist_ok=True)

required = {
    "train.csv",
    "validation.csv",
    "test.csv",
    "v2_valid.csv",
    "v2_test.csv",
    "validation_folds.csv",
    "old_xlmr_predictions.npz",
    "submission_v14_champion.csv",
}

# Case A: an actual v17_data.zip exists.
zip_matches = list(INPUT_ROOT.rglob("v17_data.zip"))

if zip_matches:
    DATA_ZIP = zip_matches[0]
    print("Found ZIP:", DATA_ZIP)
    with zipfile.ZipFile(DATA_ZIP, "r") as z:
        z.extractall(WORK_DATA)
    DATA_DIR = WORK_DATA

else:
    # Case B: Kaggle already extracted the ZIP when creating the dataset.
    print("No v17_data.zip file found. Searching for extracted V17 files...")

    candidate_dirs = []
    for train_path in INPUT_ROOT.rglob("train.csv"):
        parent = train_path.parent
        names = {p.name for p in parent.iterdir() if p.is_file()}
        if required.issubset(names):
            candidate_dirs.append(parent)

    if not candidate_dirs:
        print("\nFiles Kaggle can currently see under /kaggle/input:")
        for p in sorted(INPUT_ROOT.rglob("*")):
            if p.is_file():
                print(" ", p)
        raise FileNotFoundError(
            "Could not find the V17 dataset files. "
            "Make sure the 'v17-data' dataset is attached under Input."
        )

    # Prefer a directory whose path contains v17.
    candidate_dirs.sort(key=lambda p: ("v17" not in str(p).lower(), len(str(p))))
    DATA_DIR = candidate_dirs[0]
    print("Found extracted dataset folder:", DATA_DIR)

# Final verification.
missing = [name for name in sorted(required) if not (DATA_DIR / name).exists()]
assert not missing, f"Missing required files: {missing}"

print("\n✅ V17 data ready.")
print("DATA_DIR =", DATA_DIR)
print("Files:")
for name in sorted(required):
    p = DATA_DIR / name
    print(f"  {name:32s} {p.stat().st_size:,} bytes")

In [ ]:
# 2. Check/install dependencies
import importlib, subprocess, sys
need=[]
for mod,pkg in [("transformers","transformers>=4.45"),("sentencepiece","sentencepiece"),("sklearn","scikit-learn")]:
    try: importlib.import_module(mod)
    except Exception: need.append(pkg)
if need:
    print("Installing",need)
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*need])
else: print("Dependencies already available.")


In [ ]:
from pathlib import Path
WORKER=Path('/kaggle/working/v17_worker.py')
WORKER.write_text('\nimport argparse, gc, re, math, random, unicodedata\nfrom pathlib import Path\nimport numpy as np, pandas as pd, torch\nfrom torch.utils.data import Dataset, DataLoader\nfrom tqdm.auto import tqdm\nfrom transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup\n\nap=argparse.ArgumentParser()\nap.add_argument("--model",required=True)\nap.add_argument("--name",required=True)\nap.add_argument("--view",required=True)\nap.add_argument("--data",required=True)\nap.add_argument("--out",required=True)\na=ap.parse_args()\n\nDATA=Path(a.data); OUT=Path(a.out); OUT.mkdir(parents=True,exist_ok=True)\nSEED=42; NF=5; TARGET="is_toxic"\nLABELS=["is_toxic","implicit_toxicity","sarcasm","contains_profanity","threatening","identity_attack","sexual_content"]\nEPOCHS=4; SNAP=[3,4]; BS=8; ACC=2; EBS=32\nfold_seeds=[42,3407,2026,17,1337]\n\ndef seed(s):\n    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)\n    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False\n\nassert torch.cuda.is_available()\nprint(a.name,"GPU:",torch.cuda.get_device_name(0),flush=True)\n\ntrain=pd.read_csv(DATA/"train.csv",dtype={"ID":str})\nvalid=pd.read_csv(DATA/"validation.csv",dtype={"ID":str})\ntest=pd.read_csv(DATA/"test.csv",dtype={"ID":str})\ncombined=pd.concat([train,valid],ignore_index=True)\n\ndef clean(x): return re.sub(r"\\s+"," ",unicodedata.normalize("NFC",str(x))).strip()\ndef v2(x): return re.sub(r"\\s+"," ",str(x).replace("\\u200c"," ").replace("\\u200d"," ")).strip()\ndef group(x): return re.sub(r"\\s+"," ",unicodedata.normalize("NFKC",str(x))).strip().lower()\n\nfor f in [combined,test]:\n    f["clean_text"]=f["text"].map(clean); f["v2_text"]=f["text"].map(v2)\ncombined["group_text"]=combined["text"].map(group)\n\ngs=combined.groupby("group_text",sort=False).agg(size=("ID","size"),**{f"sum_{l}":(l,"sum") for l in LABELS}).reset_index()\ngp=np.array([combined[l].sum() for l in LABELS],float); total=float(len(combined))\nrarity=1/np.sqrt(np.maximum(gp,1)); mat=gs[[f"sum_{l}" for l in LABELS]].to_numpy(float)\nrng=np.random.default_rng(SEED); priority=(mat*rarity).sum(1)+.005*gs["size"].to_numpy()\norder=np.argsort(-(priority+rng.random(len(gs))*1e-6))\nfp=np.zeros((NF,len(LABELS))); fs=np.zeros(NF); assign={}\nfor idx in order:\n    row=gs.iloc[idx]; vec=np.array([row[f"sum_{l}"] for l in LABELS],float); size=float(row["size"])\n    costs=[]\n    for fold in range(NF):\n        p=fp.copy(); s=fs.copy(); p[fold]+=vec; s[fold]+=size\n        costs.append(np.mean((p/np.maximum(gp[None,:],1)-1/NF)**2)+.35*np.mean((s/total-1/NF)**2))\n    fold=int(np.argmin(costs)); assign[row["group_text"]]=fold; fp[fold]+=vec; fs[fold]+=size\ncombined["fold"]=combined["group_text"].map(assign).astype(int)\ncounts=combined.groupby("fold").size().reindex(range(NF),fill_value=0)\nassert (counts>0).all() and combined.groupby("group_text")["fold"].nunique().max()==1\nprint(a.name,"fold sizes",counts.to_dict(),flush=True)\n\nclass DS(Dataset):\n    def __init__(self,df,tok,labels=True):\n        self.x=df[a.view].tolist(); self.y=df[TARGET].to_numpy(np.float32) if labels else None; self.tok=tok\n    def __len__(self): return len(self.x)\n    def __getitem__(self,i):\n        z=self.tok(self.x[i],truncation=True,max_length=128,padding=False)\n        if self.y is not None: z["labels"]=self.y[i]\n        return z\n\nclass Col:\n    def __init__(self,tok): self.tok=tok\n    def __call__(self,items):\n        y=None\n        if "labels" in items[0]: y=torch.tensor([x.pop("labels") for x in items],dtype=torch.float32)\n        b=self.tok.pad(items,padding=True,pad_to_multiple_of=8,return_tensors="pt")\n        if y is not None: b["labels"]=y\n        return b\n\n@torch.no_grad()\ndef pred(model,loader):\n    model.eval(); out=[]\n    for batch in loader:\n        batch.pop("labels",None); inp={k:v.cuda(non_blocking=True) for k,v in batch.items()}\n        with torch.autocast("cuda",dtype=torch.float16):\n            z=model(**inp).logits.squeeze(-1)\n        out.append(torch.sigmoid(z.float()).cpu().numpy())\n    return np.concatenate(out)\n\noof=np.zeros(len(combined),np.float32); tests=[]; history=[]\nfor fold in range(NF):\n    tr=combined[combined.fold!=fold].copy(); va=combined[combined.fold==fold].copy()\n    cache=OUT/f"{a.name}_fold_{fold}.npz"; hist=OUT/f"{a.name}_fold_{fold}_history.csv"\n    if cache.exists():\n        with np.load(cache,allow_pickle=False) as z: idx=z["va_index"]; vp=z["valid"]; tp=z["test"]\n        if np.array_equal(idx,va.index.to_numpy()) and len(vp)==len(va) and len(tp)==len(test):\n            print(a.name,"reuse fold",fold,flush=True); oof[idx]=vp; tests.append(tp); continue\n\n    seed(fold_seeds[fold]); print(a.name,"fold",fold,"train",len(tr),"valid",len(va),flush=True)\n    tok=AutoTokenizer.from_pretrained(a.model); col=Col(tok)\n    tl=DataLoader(DS(tr,tok),batch_size=BS,shuffle=True,num_workers=2,pin_memory=True,collate_fn=col)\n    vl=DataLoader(DS(va,tok),batch_size=EBS,shuffle=False,num_workers=2,pin_memory=True,collate_fn=col)\n    el=DataLoader(DS(test,tok,False),batch_size=EBS,shuffle=False,num_workers=2,pin_memory=True,collate_fn=col)\n    model=AutoModelForSequenceClassification.from_pretrained(a.model,num_labels=1).cuda()\n    pos=float(tr[TARGET].sum()); neg=len(tr)-pos\n    lossfn=torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([np.sqrt(neg/max(pos,1))],device="cuda"))\n    opt=torch.optim.AdamW(model.parameters(),lr=1.5e-5,weight_decay=.01)\n    steps=math.ceil(len(tl)/ACC)*EPOCHS\n    sch=get_linear_schedule_with_warmup(opt,int(steps*.10),steps); scaler=torch.amp.GradScaler("cuda")\n    vs=[]; ts=[]; fh=[]\n    for epoch in range(1,EPOCHS+1):\n        model.train(); opt.zero_grad(set_to_none=True); run=seen=0\n        for step,b in enumerate(tqdm(tl,desc=f"{a.name} f{fold} e{epoch}/{EPOCHS}")):\n            y=b.pop("labels").cuda(non_blocking=True); inp={k:v.cuda(non_blocking=True) for k,v in b.items()}\n            with torch.autocast("cuda",dtype=torch.float16): logits=model(**inp).logits.squeeze(-1)\n            loss=lossfn(logits.float(),y); scaler.scale(loss/ACC).backward()\n            run+=float(loss.item())*len(y); seen+=len(y)\n            if (step+1)%ACC==0 or step+1==len(tl):\n                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1)\n                old=scaler.get_scale(); scaler.step(opt); scaler.update()\n                if scaler.get_scale()>=old: sch.step()\n                opt.zero_grad(set_to_none=True)\n        rec={"model":a.name,"fold":fold,"epoch":epoch,"loss":run/max(seen,1)}\n        print(rec,flush=True); fh.append(rec); pd.DataFrame(fh).to_csv(hist,index=False)\n        if epoch in SNAP: vs.append(pred(model,vl)); ts.append(pred(model,el))\n    vp=np.mean(vs,0); tp=np.mean(ts,0); oof[va.index.to_numpy()]=vp; tests.append(tp); history+=fh\n    np.savez_compressed(cache,va_index=va.index.to_numpy(),valid=vp,test=tp)\n    del model,tok,tl,vl,el,opt,sch,scaler; gc.collect(); torch.cuda.empty_cache()\n\nnp.savez_compressed(OUT/f"{a.name}_oof_test.npz",oof=oof,test=np.mean(tests,0))\npd.DataFrame(history).to_csv(OUT/f"{a.name}_training_history.csv",index=False)\nprint(a.name,"DONE",flush=True)\n',encoding='utf-8')
print('worker bytes',WORKER.stat().st_size)

In [ ]:
# 4. Launch one model family on each GPU at the same time
import os, sys, subprocess, time, torch
from pathlib import Path
RUN_DIR=Path("/kaggle/working/v17_output"); RUN_DIR.mkdir(exist_ok=True)
print("GPU count:",torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(i,torch.cuda.get_device_name(i))
assert torch.cuda.device_count()>=2, "Select GPU T4 x2 in Kaggle Settings."

common=[sys.executable,str(WORKER),"--data",str(DATA_DIR),"--out",str(RUN_DIR)]
env0=os.environ.copy(); env0["CUDA_VISIBLE_DEVICES"]="0"
env1=os.environ.copy(); env1["CUDA_VISIBLE_DEVICES"]="1"
log0=open("/kaggle/working/bangla_gpu0.log","w")
log1=open("/kaggle/working/xlmr_gpu1.log","w")
p0=subprocess.Popen(common+["--model","csebuetnlp/banglabert","--name","banglabert_toxic","--view","v2_text"],env=env0,stdout=log0,stderr=subprocess.STDOUT)
p1=subprocess.Popen(common+["--model","FacebookAI/xlm-roberta-base","--name","xlmr_toxic","--view","clean_text"],env=env1,stdout=log1,stderr=subprocess.STDOUT)
start=time.time()
while p0.poll() is None or p1.poll() is None:
    print(f"{(time.time()-start)/60:.1f} min | Bangla={p0.poll()} | XLM-R={p1.poll()}")
    subprocess.run(["tail","-n","2","/kaggle/working/bangla_gpu0.log"],check=False)
    subprocess.run(["tail","-n","2","/kaggle/working/xlmr_gpu1.log"],check=False)
    time.sleep(60)
log0.close(); log1.close()
print("exit codes",p0.returncode,p1.returncode)
if p0.returncode: subprocess.run(["tail","-n","80","/kaggle/working/bangla_gpu0.log"])
if p1.returncode: subprocess.run(["tail","-n","80","/kaggle/working/xlmr_gpu1.log"])
assert p0.returncode==0 and p1.returncode==0
print("Both workers finished.")


In [ ]:
# 5. Load predictions and reconstruct V14 toxic baseline
import numpy as np, pandas as pd, json
from sklearn.metrics import f1_score
train=pd.read_csv(DATA_DIR/"train.csv",dtype={"ID":str})
valid=pd.read_csv(DATA_DIR/"validation.csv",dtype={"ID":str})
test=pd.read_csv(DATA_DIR/"test.csv",dtype={"ID":str})
v2v=pd.read_csv(DATA_DIR/"v2_valid.csv",dtype={"ID":str})
v2t=pd.read_csv(DATA_DIR/"v2_test.csv",dtype={"ID":str})
vf=pd.read_csv(DATA_DIR/"validation_folds.csv",dtype={"ID":str})
champ=pd.read_csv(DATA_DIR/"submission_v14_champion.csv",dtype={"ID":str})
with np.load(DATA_DIR/"old_xlmr_predictions.npz") as z: oxv=z["valid"].copy(); oxt=z["test"].copy()
with np.load(RUN_DIR/"banglabert_toxic_oof_test.npz") as z: bo=z["oof"].copy(); bt=z["test"].copy()
with np.load(RUN_DIR/"xlmr_toxic_oof_test.npz") as z: xo=z["oof"].copy(); xt=z["test"].copy()
basev=.875*v2v["is_toxic_prob"].to_numpy(float)+.125*oxv[1:3,:,0].mean(0)
baset=.875*v2t["is_toxic_prob"].to_numpy(float)+.125*oxt[1:3,:,0].mean(0)
idx=np.arange(len(train),len(train)+len(valid))
bv=bo[idx]; xv=xo[idx]
folds=valid["ID"].map(vf.set_index("ID")["fold"]).to_numpy(int)
y=valid["is_toxic"].to_numpy(int)
def logit(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6); return np.log(p/(1-p))
def sig(z): return 1/(1+np.exp(-z))
def macro(y,p): return float(f1_score(y,p,average="macro",labels=[0,1],zero_division=0))
def best_t(y,p):
    vals=np.unique(p)
    if len(vals)>600: vals=np.quantile(vals,np.linspace(0,1,600))
    cand=np.unique(np.r_[0.,vals,1.000001]); s=np.array([macro(y,p>=t) for t in cand])
    return float(np.median(cand[np.isclose(s,s.max(),atol=1e-12,rtol=0)]))
def cv(p):
    pred=np.zeros(len(y),np.int8); fs=[]; ts=[]
    for f in range(5):
        h=folds==f; t=best_t(y[~h],p[~h]); ts.append(t); pred[h]=(p[h]>=t).astype(np.int8); fs.append(macro(y[h],pred[h]))
    return macro(y,pred),np.array(fs),np.array(ts)
base_score,base_fs,base_ts=cv(basev)
print("V14 toxic baseline:",base_score,base_fs)


In [ ]:
# 6. Search conservative 3-way ensembles
rows=[]; best=None
for mode in ["prob","logit"]:
    for wb_i in range(8,21):
        wb=wb_i/20; rem=1-wb
        for k in range(int(round(rem*20))+1):
            wbang=k/20; wx=rem-wbang
            if wx<-1e-12: continue
            p=wb*basev+wbang*bv+wx*xv if mode=="prob" else sig(wb*logit(basev)+wbang*logit(bv)+wx*logit(xv))
            score,fs,ts=cv(p)
            wins=int((fs>base_fs+1e-12).sum()); losses=int((fs<base_fs-1e-12).sum()); ties=5-wins-losses
            gain=score-base_score; passes=gain>=.003 and wins>=3 and losses<=1
            rec={"mode":mode,"w_baseline":wb,"w_banglabert":wbang,"w_xlmr":wx,"score":score,"gain":gain,"wins":wins,"ties":ties,"losses":losses,"passes_gate":passes,"threshold":float(np.median(ts))}
            rows.append(rec); key=(passes,score,wins,-losses,wb)
            if best is None or key>best[0]: best=(key,rec,fs,ts)
search=pd.DataFrame(rows).sort_values(["passes_gate","gain","wins","losses"],ascending=[False,False,False,True])
search.to_csv(RUN_DIR/"toxic_ensemble_search.csv",index=False)
display(search.head(25))
print("BEST",best[1])


In [ ]:
# 7. Build submission only if the gate passes
rec=best[1]; passes=bool(rec["passes_gate"])
if rec["mode"]=="prob":
    pt=rec["w_baseline"]*baset+rec["w_banglabert"]*bt+rec["w_xlmr"]*xt
else:
    pt=sig(rec["w_baseline"]*logit(baset)+rec["w_banglabert"]*logit(bt)+rec["w_xlmr"]*logit(xt))
pred=(pt>=rec["threshold"]).astype(np.int8)
sub=champ.copy(); before=sub["is_toxic"].to_numpy(int)
if passes: sub["is_toxic"]=pred
changed=int((before!=sub["is_toxic"].to_numpy(int)).sum())
sub.to_csv(RUN_DIR/"submission_v17_kaggle_toxic.csv",index=False)
report={"champion_score":0.586,"baseline_toxic_cv":base_score,"selected":rec,"passes_gate":passes,"changed_rows":changed}
(RUN_DIR/"v17_report.json").write_text(json.dumps(report,indent=2))
print(report)


In [ ]:
# 8. Package results
import zipfile
zip_path=Path("/kaggle/working/v17_results.zip")
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file(): z.write(p,arcname=p.name)
    for p in [Path("/kaggle/working/bangla_gpu0.log"),Path("/kaggle/working/xlmr_gpu1.log")]:
        if p.exists(): z.write(p,arcname=p.name)
print("Created",zip_path)
print("Download it from Kaggle's Files/Output panel and send it to ChatGPT.")
